# Hakbang PH — Skills-first career recommender

This Google Colab notebook trains a career-job-family classifier on
**2,200 deterministic synthetic profiles**. The only user features are:

- total years of work experience;
- 12 self-assessed, demonstrated skill ratings from 0 to 5.

Industry, current job title, employer, career goal, demographics, and
protected characteristics are excluded. Research claims, certifications,
courses, and practitioner accounts come from a fixed source ledger; the
notebook does not generate factual claims.

**Important:** synthetic cross-validation measures recovery of synthetic
labels. It is not proof of real-world hiring, salary, or career success.

In [ ]:
!pip -q install "numpy>=1.26,<3" "pandas>=2,<3" "scikit-learn>=1.4,<2"

In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260725
RNG = np.random.default_rng(SEED)

## Fixed evidence and job-family catalog

The catalog below is exported from the same registry used by the app.
Every demand statement includes its original source link. Every learning
option links to the official provider. Practitioner accounts are labeled
as anecdotes or issuer surveys and include a causal caveat.

In [ ]:
CATALOG = {
  "datasetVersion": "PH-SKILLS-FIRST-SYN-2200-2026.07.28",
  "sampleCount": 2200,
  "modelName": "Extra Trees",
  "syntheticMacroF1": 0.7854,
  "syntheticAccuracy": 0.7868,
  "evidenceChecked": "28 July 2026",
  "syntheticDataStatement": "Every profile and label is generated from documented job-family prototypes. The dataset contains no real person, résumé, employer record, protected characteristic, or vacancy.",
  "features": [
    "years_experience",
    "data_analytics",
    "ai_automation",
    "software_cloud",
    "cybersecurity_risk",
    "communication",
    "project_change",
    "creative_design",
    "finance_commercial",
    "people_coaching",
    "operations_quality",
    "customer_research",
    "scientific_laboratory"
  ],
  "excludedFeatures": [
    "industry",
    "current_job_title",
    "employer",
    "career_goal",
    "age",
    "sex",
    "gender",
    "race",
    "religion",
    "disability",
    "marital_status"
  ],
  "weights": {
    "skillAlignment": 0.5,
    "coreSkillCoverage": 0.15,
    "syntheticModelFit": 0.15,
    "experienceProximity": 0.08,
    "currentDemand": 0.05,
    "futureDemand": 0.07
  },
  "supportThresholds": {
    "skillAlignment": 0.4,
    "coreSkillCoverage": 0.25,
    "comparativeScore": 55
  },
  "skills": [
    {
      "key": "data_analytics",
      "label": "Data & analytics",
      "help": "Spreadsheets, reporting, SQL, statistics, dashboards, and evidence"
    },
    {
      "key": "ai_automation",
      "label": "AI & automation",
      "help": "Responsible AI use, prompting, workflow automation, and output checking"
    },
    {
      "key": "software_cloud",
      "label": "Software & cloud",
      "help": "Applications, systems, cloud platforms, coding, and troubleshooting"
    },
    {
      "key": "cybersecurity_risk",
      "label": "Cybersecurity & risk",
      "help": "Security controls, privacy, governance, audit, and risk response"
    },
    {
      "key": "communication",
      "label": "Communication & storytelling",
      "help": "Writing, presenting, negotiation, facilitation, and explaining decisions"
    },
    {
      "key": "project_change",
      "label": "Project & change delivery",
      "help": "Planning, scope, risk, coordination, adoption, and improvement delivery"
    },
    {
      "key": "creative_design",
      "label": "Creative & design",
      "help": "Design, campaigns, prototyping, content, and ideation"
    },
    {
      "key": "finance_commercial",
      "label": "Finance & commercial",
      "help": "Budgeting, accounting, forecasting, pricing, and commercial analysis"
    },
    {
      "key": "people_coaching",
      "label": "People & coaching",
      "help": "Coaching, talent, teamwork, workforce planning, and employee experience"
    },
    {
      "key": "operations_quality",
      "label": "Operations & quality",
      "help": "Process improvement, logistics, controls, quality, and service delivery"
    },
    {
      "key": "customer_research",
      "label": "Customer & user research",
      "help": "Research, service design, customer success, usability, and voice of customer"
    },
    {
      "key": "scientific_laboratory",
      "label": "Scientific & laboratory practice",
      "help": "Experimental methods, analytical chemistry, laboratory quality, validation, and safety"
    }
  ],
  "careers": [
    {
      "id": "data_bi_analyst",
      "title": "Data & Business Intelligence Analyst",
      "summary": "Turn operational and commercial data into dashboards, decisions, and measurable business improvements.",
      "experienceTarget": 4.0,
      "skills": {
        "data_analytics": 0.98,
        "ai_automation": 0.78,
        "software_cloud": 0.6,
        "cybersecurity_risk": 0.35,
        "communication": 0.72,
        "project_change": 0.6,
        "creative_design": 0.25,
        "finance_commercial": 0.55,
        "people_coaching": 0.25,
        "operations_quality": 0.5,
        "customer_research": 0.45,
        "scientific_laboratory": 0.2
      },
      "coreSkills": [
        "data_analytics"
      ],
      "applicationContexts": [
        "Finance and insurance",
        "Healthcare and life sciences",
        "Retail and e-commerce",
        "Government and public services",
        "Technology and professional services"
      ],
      "currentDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Sector-to-role inference",
        "insight": "DOLE identifies IT–BPM/BPO as an in-demand Philippine sector, while TESDA's 2025 skills report names big-data work among high-growth occupations. The BI mapping is an inference, not a vacancy count.",
        "sources": [
          {
            "name": "Jobs and Labor Market Forecast",
            "owner": "DOLE Bureau of Local Employment",
            "url": "https://ble.dole.gov.ph/jobs-and-labor-market-forecast/",
            "published": "2023–2025 release"
          },
          {
            "name": "TVET Skills Insights: 5th Industrial Revolution",
            "owner": "TESDA",
            "url": "https://www.tesda.gov.ph/Uploads/File/SkillInsights/2025/TVET%20Skills%20Insights%20Report%20_%205th%20Industrial%20Revolution.pdf",
            "published": "2025"
          }
        ]
      },
      "futureDemand": {
        "label": "Very strong",
        "score": 1.0,
        "basis": "Global directional evidence",
        "insight": "WEF's employer survey places Big Data Specialists first among the fastest-growing roles through 2030 and AI and big data first among rising skills.",
        "sources": [
          {
            "name": "Future of Jobs 2025 — Jobs Outlook",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/2-jobs-outlook/",
            "published": "8 January 2025"
          },
          {
            "name": "Future of Jobs 2025 — Skills Outlook",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/3-skills-outlook/",
            "published": "8 January 2025"
          }
        ]
      },
      "aiOpportunity": "Use copilots to draft measures, explain variance, and accelerate exploration, then own metric definitions, data quality, and decision context.",
      "humanEdge": "Stakeholder framing and checking whether a statistically correct output answers the real business question.",
      "firstProof": "Build one decision dashboard with a documented data dictionary, AI-use log, and before/after business metric.",
      "certificationEvidence": {
        "name": "Microsoft Certified: Power BI Data Analyst Associate",
        "issuer": "Microsoft",
        "url": "https://learn.microsoft.com/en-us/credentials/certifications/data-analyst-associate/",
        "eligibility": "Intermediate credential; review the official PL-300 exam page.",
        "why_it_fits": "PL-300 assesses data preparation, modeling, visualization, analysis, management, and security in Power BI.",
        "practitioner": "Sarah Krusleski · Power BI practitioner",
        "practitioner_insight": "After earning PL-300, she reported recruiter messages for roles that explicitly required it and inquiries about teaching Power BI. She also says certification is not mandatory.",
        "source_type": "Public first-person account",
        "practitioner_url": "https://www.linkedin.com/posts/sekrusleski_what-will-passing-the-pl-300-microsoft-power-activity-7269734663479279616-quT4",
        "caveat": "A credential can validate knowledge; it does not replace role-relevant projects, supervised practice, or measurable work outcomes."
      },
      "learningOptions": [
        {
          "type": "Certification",
          "name": "Microsoft Certified: Power BI Data Analyst Associate",
          "provider": "Microsoft",
          "url": "https://learn.microsoft.com/en-us/credentials/certifications/data-analyst-associate/",
          "fit": "Validates preparing, modeling, visualizing, analyzing, and securing data in Power BI.",
          "eligibility": "Intermediate credential; Microsoft lists no formal work-experience prerequisite. Review the current PL-300 study guide."
        },
        {
          "type": "Course",
          "name": "PL-300: Design and manage analytics solutions using Power BI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/pl-300t00",
          "fit": "Official preparation covering the Power BI workflow assessed by PL-300.",
          "eligibility": "Review the course prerequisites and current delivery options on Microsoft Learn."
        },
        {
          "type": "AI course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "A beginner, no-code course on applying generative AI to workflows, decisions, and business outcomes.",
          "eligibility": "Designed for business users across functions; confirm current product-access and delivery requirements."
        }
      ]
    },
    {
      "id": "cybersecurity_analyst",
      "title": "Cybersecurity Analyst",
      "summary": "Monitor threats, investigate incidents, strengthen controls, and help organizations manage digital risk.",
      "experienceTarget": 3.5,
      "skills": {
        "data_analytics": 0.65,
        "ai_automation": 0.7,
        "software_cloud": 0.9,
        "cybersecurity_risk": 0.98,
        "communication": 0.65,
        "project_change": 0.65,
        "creative_design": 0.2,
        "finance_commercial": 0.25,
        "people_coaching": 0.25,
        "operations_quality": 0.7,
        "customer_research": 0.3,
        "scientific_laboratory": 0.15
      },
      "coreSkills": [
        "cybersecurity_risk",
        "software_cloud"
      ],
      "applicationContexts": [
        "Banking and financial services",
        "Government and critical infrastructure",
        "Healthcare",
        "Telecommunications",
        "Technology and business-process services"
      ],
      "currentDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Direct role evidence",
        "insight": "ISC2 reports entry-level CC holders in analyst and security-operations pathways; DOLE separately identifies IT–BPM/BPO as an in-demand Philippine sector.",
        "sources": [
          {
            "name": "Who earns the ISC2 CC?",
            "owner": "ISC2",
            "url": "https://www.isc2.org/insights/2025/11/who-earns-the-isc2-certified-in-cybersecurity-certification",
            "published": "3 November 2025"
          },
          {
            "name": "Jobs and Labor Market Forecast",
            "owner": "DOLE Bureau of Local Employment",
            "url": "https://ble.dole.gov.ph/jobs-and-labor-market-forecast/",
            "published": "2023–2025 release"
          }
        ]
      },
      "futureDemand": {
        "label": "Very strong",
        "score": 1.0,
        "basis": "Global directional evidence",
        "insight": "WEF lists Information Security Analysts among the 15 fastest-growing roles and networks and cybersecurity as the second-fastest-rising skill group through 2030.",
        "sources": [
          {
            "name": "Future of Jobs 2025 — Jobs Outlook",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/2-jobs-outlook/",
            "published": "8 January 2025"
          },
          {
            "name": "Future of Jobs 2025 — Skills Outlook",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/3-skills-outlook/",
            "published": "8 January 2025"
          }
        ]
      },
      "aiOpportunity": "Apply AI to alert triage, threat research, and control documentation while testing AI systems for prompt injection, data leakage, and model abuse.",
      "humanEdge": "Risk judgment, incident accountability, and adversarial reasoning when automated output is incomplete or misleading.",
      "firstProof": "Create an incident-response lab and publish a redacted write-up covering detection, evidence, containment, and lessons learned.",
      "certificationEvidence": {
        "name": "ISC2 Certified in Cybersecurity (CC)",
        "issuer": "ISC2",
        "url": "https://www.isc2.org/certifications/cc",
        "eligibility": "No prior work experience is required. Review the official page for the current exam outline.",
        "why_it_fits": "CC validates foundational security principles for an entry or adjacent career move.",
        "practitioner": "Lance Rosengarten, CC · SOC Analyst",
        "practitioner_insight": "He wrote that, weeks after passing CC, he entered a GRC internship and was later offered a SOC Analyst Team Lead role.",
        "source_type": "Issuer-published holder account",
        "practitioner_url": "https://www.isc2.org/Insights/2024/02/My-Journey-into-Cybersecurity-With-ISC2",
        "caveat": "This sequence does not prove that CC caused the outcome. He also used labs, self-study, and an internship."
      },
      "learningOptions": [
        {
          "type": "Certification",
          "name": "Certified in Cybersecurity (CC)",
          "provider": "ISC2",
          "url": "https://www.isc2.org/certifications/cc",
          "fit": "Entry-level coverage of security principles, incident response, access controls, networks, and security operations.",
          "eligibility": "ISC2 states that no work experience is required. Confirm current exam, training, and membership terms."
        },
        {
          "type": "Official training",
          "name": "ISC2 CC Online Self-Paced Training",
          "provider": "ISC2",
          "url": "https://www.isc2.org/certifications/cc",
          "fit": "Official training aligned with the current CC exam domains.",
          "eligibility": "Availability and pricing can change; verify them on the ISC2 page before enrolling."
        },
        {
          "type": "AI course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "A beginner, no-code course on applying generative AI to workflows, decisions, and business outcomes.",
          "eligibility": "Designed for business users across functions; confirm current product-access and delivery requirements."
        }
      ]
    },
    {
      "id": "cloud_solutions_engineer",
      "title": "Cloud Solutions Engineer",
      "summary": "Design reliable cloud environments and improve the cost, security, and performance of digital systems.",
      "experienceTarget": 4.5,
      "skills": {
        "data_analytics": 0.6,
        "ai_automation": 0.8,
        "software_cloud": 0.98,
        "cybersecurity_risk": 0.75,
        "communication": 0.65,
        "project_change": 0.72,
        "creative_design": 0.25,
        "finance_commercial": 0.25,
        "people_coaching": 0.25,
        "operations_quality": 0.75,
        "customer_research": 0.3,
        "scientific_laboratory": 0.15
      },
      "coreSkills": [
        "software_cloud",
        "ai_automation"
      ],
      "applicationContexts": [
        "Technology and telecommunications",
        "Banking and financial services",
        "Retail and e-commerce",
        "Government digital services",
        "Professional and managed services"
      ],
      "currentDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Sector-to-role inference",
        "insight": "DOLE identifies IT–BPM/BPO as in demand. Cloud engineering is treated as enabling infrastructure, not counted as a live national vacancy total.",
        "sources": [
          {
            "name": "Jobs and Labor Market Forecast",
            "owner": "DOLE Bureau of Local Employment",
            "url": "https://ble.dole.gov.ph/jobs-and-labor-market-forecast/",
            "published": "2023–2025 release"
          },
          {
            "name": "TVET Skills Insights: 5th Industrial Revolution",
            "owner": "TESDA",
            "url": "https://www.tesda.gov.ph/Uploads/File/SkillInsights/2025/TVET%20Skills%20Insights%20Report%20_%205th%20Industrial%20Revolution.pdf",
            "published": "2025"
          }
        ]
      },
      "futureDemand": {
        "label": "Very strong",
        "score": 1.0,
        "basis": "Sector-to-role inference",
        "insight": "WEF says information and technology services employers expect near-universal AI and information-processing adoption by 2030. Cloud-architecture demand is an inference from that adoption.",
        "sources": [
          {
            "name": "Future of Jobs 2025 — Industry Insights",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/5-region-economy-and-industry-insights/",
            "published": "8 January 2025"
          }
        ]
      },
      "aiOpportunity": "Design the governed data, identity, security, observability, and cost controls that let teams run AI workloads reliably.",
      "humanEdge": "Architectural trade-offs across resilience, security, latency, cost, and regulation.",
      "firstProof": "Deploy a retrieval-based AI service with least-privilege access, monitoring, a cost budget, and an architecture decision record.",
      "certificationEvidence": {
        "name": "AWS Certified Solutions Architect – Associate",
        "issuer": "Amazon Web Services",
        "url": "https://aws.amazon.com/certification/certified-solutions-architect-associate/",
        "eligibility": "AWS recommends prior hands-on experience; review the official exam guide.",
        "why_it_fits": "The credential validates design of secure, resilient, high-performing, and cost-optimized AWS solutions.",
        "practitioner": "Siddharth Pasumarthy · AWS Solutions Architect",
        "practitioner_insight": "He says hands-on labs broadened his technical range; after more than a year of experience and the certification, he accepted an AWS Solutions Architect offer.",
        "source_type": "Issuer-published holder account",
        "practitioner_url": "https://aws.amazon.com/blogs/training-and-certification/steps-to-start-your-aws-certification-journey/",
        "caveat": "The holder had substantial self-learning and hands-on platform experience; the credential was one part of the transition."
      },
      "learningOptions": [
        {
          "type": "Certification",
          "name": "AWS Certified Solutions Architect – Associate",
          "provider": "Amazon Web Services",
          "url": "https://aws.amazon.com/certification/certified-solutions-architect-associate/",
          "fit": "Covers secure, resilient, high-performing, and cost-optimized cloud solution design.",
          "eligibility": "No formal prerequisite; AWS recommends about one year of hands-on cloud solution-design experience."
        },
        {
          "type": "Learning path",
          "name": "Microsoft Azure Fundamentals: Describe cloud concepts",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/paths/microsoft-azure-fundamentals-describe-cloud-concepts/",
          "fit": "A beginner path for cloud concepts before deeper platform architecture study.",
          "eligibility": "Beginner learning path; check the live Microsoft Learn page for current modules."
        },
        {
          "type": "AI course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "A beginner, no-code course on applying generative AI to workflows, decisions, and business outcomes.",
          "eligibility": "Designed for business users across functions; confirm current product-access and delivery requirements."
        }
      ]
    },
    {
      "id": "project_manager",
      "title": "Project Manager",
      "summary": "Lead cross-functional work, align stakeholders, manage risk, and deliver business outcomes across industries.",
      "experienceTarget": 6.5,
      "skills": {
        "data_analytics": 0.5,
        "ai_automation": 0.6,
        "software_cloud": 0.5,
        "cybersecurity_risk": 0.35,
        "communication": 0.95,
        "project_change": 0.98,
        "creative_design": 0.5,
        "finance_commercial": 0.5,
        "people_coaching": 0.78,
        "operations_quality": 0.82,
        "customer_research": 0.7,
        "scientific_laboratory": 0.15
      },
      "coreSkills": [
        "project_change",
        "communication"
      ],
      "applicationContexts": [
        "Technology transformation",
        "Construction and infrastructure",
        "Healthcare",
        "Finance and professional services",
        "Government and nonprofit programs"
      ],
      "currentDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Global directional evidence",
        "insight": "PMI's 2025 research describes project talent as cross-industry and reports sustained global demand. This is not a Philippines-only vacancy measure.",
        "sources": [
          {
            "name": "PMP salary and talent survey",
            "owner": "Project Management Institute",
            "url": "https://www.pmi.org/about/press-media/2025/pmp-certification-holders-build-career-momentum-and-experience-earning-advantage-pmi-survey-finds",
            "published": "13 November 2025"
          }
        ]
      },
      "futureDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Global directional evidence",
        "insight": "PMI projects that the world may need up to 30 million additional project professionals by 2035 as organizations deliver AI and other transformations.",
        "sources": [
          {
            "name": "PMP salary and talent survey",
            "owner": "Project Management Institute",
            "url": "https://www.pmi.org/about/press-media/2025/pmp-certification-holders-build-career-momentum-and-experience-earning-advantage-pmi-survey-finds",
            "published": "13 November 2025"
          }
        ]
      },
      "aiOpportunity": "Use AI for draft plans, status synthesis, risk prompts, and meeting follow-through while keeping humans accountable for scope, value, and escalation.",
      "humanEdge": "Negotiation, change leadership, judgment under uncertainty, and ownership of outcomes.",
      "firstProof": "Lead a small AI-enabled process change with a benefits baseline, risk register, adoption plan, and post-implementation review.",
      "certificationEvidence": {
        "name": "Project Management Professional (PMP)®",
        "issuer": "Project Management Institute",
        "url": "https://www.pmi.org/certifications/project-management-pmp",
        "eligibility": "Experience and training requirements apply; consider CAPM if you are not yet eligible.",
        "why_it_fits": "PMP tests people, process, and business-environment capabilities across predictive, agile, and hybrid delivery.",
        "practitioner": "14,628 project professionals · PMI salary survey",
        "practitioner_insight": "PMI's 2025 survey across 21 countries reported a 17% higher median salary for PMP holders than non-certified respondents.",
        "source_type": "Issuer survey",
        "practitioner_url": "https://www.pmi.org/about/press-media/2025/pmp-certification-holders-build-career-momentum-and-experience-earning-advantage-pmi-survey-finds",
        "caveat": "This is an association, not proof that PMP caused the pay difference. Experience, role, geography, and employer may contribute."
      },
      "learningOptions": [
        {
          "type": "Certification",
          "name": "Certified Associate in Project Management (CAPM)®",
          "provider": "Project Management Institute",
          "url": "https://www.pmi.org/certifications/certified-associate-capm",
          "fit": "Builds foundational knowledge in predictive, agile, and business-analysis ways of working.",
          "eligibility": "PMI requires a secondary degree or equivalent plus 23 hours of project-management education; no work experience is required."
        },
        {
          "type": "Advanced certification",
          "name": "Project Management Professional (PMP)®",
          "provider": "Project Management Institute",
          "url": "https://www.pmi.org/certifications/project-management-pmp",
          "fit": "For professionals who can document substantial responsibility for leading projects.",
          "eligibility": "Experience and training requirements vary by education. Verify the current PMI eligibility route before applying."
        },
        {
          "type": "AI course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "A beginner, no-code course on applying generative AI to workflows, decisions, and business outcomes.",
          "eligibility": "Designed for business users across functions; confirm current product-access and delivery requirements."
        }
      ]
    },
    {
      "id": "digital_marketing_strategist",
      "title": "Digital Marketing & E-commerce Strategist",
      "summary": "Combine audience insight, content, paid media, and performance data to grow digital revenue.",
      "experienceTarget": 3.5,
      "skills": {
        "data_analytics": 0.68,
        "ai_automation": 0.72,
        "software_cloud": 0.58,
        "cybersecurity_risk": 0.3,
        "communication": 0.88,
        "project_change": 0.68,
        "creative_design": 0.96,
        "finance_commercial": 0.32,
        "people_coaching": 0.42,
        "operations_quality": 0.5,
        "customer_research": 0.9,
        "scientific_laboratory": 0.1
      },
      "coreSkills": [
        "creative_design",
        "communication",
        "customer_research"
      ],
      "applicationContexts": [
        "Retail and e-commerce",
        "Consumer goods",
        "Tourism and hospitality",
        "Technology and media",
        "Education and professional services"
      ],
      "currentDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Sector-to-role inference",
        "insight": "DOLE identifies services and tourism among in-demand Philippine sectors. Digital-marketing demand is inferred from those sectors and e-commerce activity, not a live count of vacancies.",
        "sources": [
          {
            "name": "Jobs and Labor Market Forecast",
            "owner": "DOLE Bureau of Local Employment",
            "url": "https://ble.dole.gov.ph/jobs-and-labor-market-forecast/",
            "published": "2023–2025 release"
          }
        ]
      },
      "futureDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Global directional evidence",
        "insight": "WEF expects marketing and media skills to grow in importance through technological change, while generative AI also increases task automation and role redesign.",
        "sources": [
          {
            "name": "Future of Jobs 2025 — Skills Outlook",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/3-skills-outlook/",
            "published": "8 January 2025"
          },
          {
            "name": "Generative AI and Jobs: A Refined Global Index",
            "owner": "International Labour Organization",
            "url": "https://www.ilo.org/publications/generative-ai-and-jobs-refined-global-index-occupational-exposure",
            "published": "20 May 2025"
          }
        ]
      },
      "aiOpportunity": "Use AI for creative variants, audience research, and campaign analysis, then differentiate through positioning, experimentation, consent, and measurement.",
      "humanEdge": "Brand judgment, cultural context, customer empathy, and knowing when apparent performance is a measurement artifact.",
      "firstProof": "Run a small campaign experiment with human-reviewed AI variants, a pre-registered hypothesis, and a clean results readout.",
      "certificationEvidence": {
        "name": "Meta Certified Digital Marketing Associate",
        "issuer": "Meta Blueprint",
        "url": "https://www.facebookblueprint.com/student/path/517001-get-certified-as-digital-marketing-associate",
        "eligibility": "Entry-level certification; confirm current language and exam availability.",
        "why_it_fits": "The credential covers Meta advertising fundamentals, targeting, creative, optimization, and measurement.",
        "practitioner": "Ahmad · Meta-certified learner",
        "practitioner_insight": "His public first-person account says passing the certificate alone produced no offers; applying the learning in visible work mattered more.",
        "source_type": "Public first-person account",
        "practitioner_url": "https://medium.com/write-a-catalyst/i-passed-the-meta-certification-and-waited-for-my-life-to-change-it-didnt-until-i-did-this-1992e64e7404",
        "caveat": "The author's identity and outcome are not independently verified. The account is included as a counterexample to credential guarantees."
      },
      "learningOptions": [
        {
          "type": "Professional certificate",
          "name": "Google Digital Marketing & E-commerce Certificate",
          "provider": "Google",
          "url": "https://grow.google/certificates/digital-marketing-ecommerce/",
          "fit": "Covers campaigns, customer engagement, analytics, e-commerce, and practical AI use in marketing.",
          "eligibility": "Google describes it as foundational and requiring no previous experience."
        },
        {
          "type": "Certification",
          "name": "Meta Certified Digital Marketing Associate",
          "provider": "Meta Blueprint",
          "url": "https://www.facebookblueprint.com/student/path/517001-get-certified-as-digital-marketing-associate",
          "fit": "Covers Meta advertising fundamentals, targeting, creative, optimization, and measurement.",
          "eligibility": "Entry-level credential; confirm current exam language, delivery, and regional availability."
        },
        {
          "type": "AI course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "A beginner, no-code course on applying generative AI to workflows, decisions, and business outcomes.",
          "eligibility": "Designed for business users across functions; confirm current product-access and delivery requirements."
        }
      ]
    },
    {
      "id": "people_analytics_specialist",
      "title": "People Analytics Specialist",
      "summary": "Use workforce data to improve retention, performance, and strategic talent decisions.",
      "experienceTarget": 4.0,
      "skills": {
        "data_analytics": 0.85,
        "ai_automation": 0.65,
        "software_cloud": 0.55,
        "cybersecurity_risk": 0.42,
        "communication": 0.82,
        "project_change": 0.68,
        "creative_design": 0.4,
        "finance_commercial": 0.38,
        "people_coaching": 0.98,
        "operations_quality": 0.55,
        "customer_research": 0.6,
        "scientific_laboratory": 0.15
      },
      "coreSkills": [
        "people_coaching",
        "data_analytics"
      ],
      "applicationContexts": [
        "Large multi-industry employers",
        "Business-process services",
        "Finance and professional services",
        "Healthcare",
        "Government and education"
      ],
      "currentDemand": {
        "label": "Moderate",
        "score": 0.6,
        "basis": "Sector-to-role inference",
        "insight": "SHRM documents an active professional pathway for people analytics, but the reviewed sources do not provide a Philippine job count for this specialist title.",
        "sources": [
          {
            "name": "People Analytics Specialty Credential",
            "owner": "Society for Human Resource Management",
            "url": "https://www.shrm.org/credentials/specialty-credentials/people-analytics-credential",
            "published": "Checked 28 July 2026"
          }
        ]
      },
      "futureDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Global directional evidence",
        "insight": "WEF places talent management among the ten fastest-rising skills through 2030; combining it with data literacy supports a forward-looking HR specialization.",
        "sources": [
          {
            "name": "Future of Jobs 2025 — Skills Outlook",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/3-skills-outlook/",
            "published": "8 January 2025"
          }
        ]
      },
      "aiOpportunity": "Use AI to surface workforce patterns and skills gaps, then test for bias, privacy risk, weak proxies, and unsupported causal conclusions.",
      "humanEdge": "Employee trust, ethical interpretation, organizational context, and responsible decisions about people.",
      "firstProof": "Create an anonymized retention or skills dashboard with a privacy note, bias checks, and clearly separated correlation versus causation.",
      "certificationEvidence": {
        "name": "SHRM People Analytics Specialty Credential",
        "issuer": "Society for Human Resource Management",
        "url": "https://www.shrm.org/credentials/specialty-credentials/people-analytics-credential",
        "eligibility": "A structured program and final knowledge assessment are required.",
        "why_it_fits": "SHRM's program connects people-data literacy, metrics, analysis, and action to HR decisions.",
        "practitioner": "Cathy Evans, SHRM-SCP · HR professional",
        "practitioner_insight": "Her issuer-published testimonial describes moving from feeling intimidated by numerical storytelling to feeling more confident and motivated to go deeper.",
        "source_type": "Issuer-published holder account",
        "practitioner_url": "https://www.shrm.org/gl/shop/product.html/shrm-people-analytics-specialty-credential-p",
        "caveat": "This is issuer-selected learning feedback, not an independently measured employment outcome."
      },
      "learningOptions": [
        {
          "type": "Specialty credential",
          "name": "People Analytics Specialty Credential",
          "provider": "Society for Human Resource Management",
          "url": "https://www.shrm.org/credentials/specialty-credentials/people-analytics-credential",
          "fit": "Combines data literacy, workforce metrics, applied analysis, and communication of people insights.",
          "eligibility": "SHRM states that SHRM-CP or SHRM-SCP certification is not required; verify current package components and availability."
        },
        {
          "type": "Course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "Builds practical, no-code AI workflow skills applicable to HR reporting and decision support.",
          "eligibility": "Beginner course for business users; review current product-access requirements."
        }
      ]
    },
    {
      "id": "supply_chain_analyst",
      "title": "Supply Chain Analyst",
      "summary": "Improve inventory, planning, sourcing, and logistics decisions with process knowledge and data.",
      "experienceTarget": 4.0,
      "skills": {
        "data_analytics": 0.84,
        "ai_automation": 0.68,
        "software_cloud": 0.58,
        "cybersecurity_risk": 0.3,
        "communication": 0.68,
        "project_change": 0.72,
        "creative_design": 0.25,
        "finance_commercial": 0.58,
        "people_coaching": 0.35,
        "operations_quality": 0.98,
        "customer_research": 0.55,
        "scientific_laboratory": 0.2
      },
      "coreSkills": [
        "operations_quality",
        "data_analytics"
      ],
      "applicationContexts": [
        "Manufacturing",
        "Retail and e-commerce",
        "Transport and logistics",
        "Healthcare supply networks",
        "Food, agriculture, and consumer goods"
      ],
      "currentDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Sector-to-role inference",
        "insight": "DOLE's national forecast covers employment-generating sectors and TESDA describes continuing digital transformation in operations. The analyst-title score is a structured inference.",
        "sources": [
          {
            "name": "Jobs and Labor Market Forecast",
            "owner": "DOLE Bureau of Local Employment",
            "url": "https://ble.dole.gov.ph/jobs-and-labor-market-forecast/",
            "published": "2023–2025 release"
          },
          {
            "name": "TVET Skills Insights: 5th Industrial Revolution",
            "owner": "TESDA",
            "url": "https://www.tesda.gov.ph/Uploads/File/SkillInsights/2025/TVET%20Skills%20Insights%20Report%20_%205th%20Industrial%20Revolution.pdf",
            "published": "2025"
          }
        ]
      },
      "futureDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Global directional evidence",
        "insight": "Digitalization, risk, and planning complexity support continued demand for analytical supply-chain skills; WEF also expects resource management and operations skills to rise.",
        "sources": [
          {
            "name": "Future of Jobs 2025 — Skills Outlook",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/3-skills-outlook/",
            "published": "8 January 2025"
          },
          {
            "name": "Certified in Planning and Inventory Management",
            "owner": "Association for Supply Chain Management",
            "url": "https://www.ascm.org/learning-development/certifications-credentials/cpim/",
            "published": "Checked 28 July 2026"
          }
        ]
      },
      "aiOpportunity": "Use AI for demand scenarios, exception detection, and inventory recommendations while retaining judgment about disruption, service levels, and supplier constraints.",
      "humanEdge": "Cross-functional trade-offs, exception handling, supplier relationships, and operational accountability.",
      "firstProof": "Build a forecast-versus-actual review with an exception queue, service-level impact, and documented human override rules.",
      "certificationEvidence": {
        "name": "APICS Certified in Planning and Inventory Management (CPIM)",
        "issuer": "Association for Supply Chain Management",
        "url": "https://www.ascm.org/learning-development/certifications-credentials/cpim/",
        "eligibility": "Confirm the current exam version and learning-system options.",
        "why_it_fits": "CPIM covers planning, inventory, demand, supply, quality, continuous improvement, and technology.",
        "practitioner": "James Tilton, CPIM · Director of Materials",
        "practitioner_insight": "His issuer-published testimonial says the credential enabled him to pursue a broader path across operations, inventory, and supply chain.",
        "source_type": "Issuer-published holder account",
        "practitioner_url": "https://www.ascm.org/learning-development/certifications-credentials/cpim/",
        "caveat": "This is an issuer-selected testimonial. Treat it as a possible pathway, not an expected result."
      },
      "learningOptions": [
        {
          "type": "Certification",
          "name": "APICS Certified in Planning and Inventory Management (CPIM)",
          "provider": "Association for Supply Chain Management",
          "url": "https://www.ascm.org/learning-development/certifications-credentials/cpim/",
          "fit": "Covers strategy alignment, S&OP, demand, supply, inventory, quality improvement, and technology.",
          "eligibility": "Review the current CPIM exam version, bundle, testing, and maintenance requirements with ASCM."
        },
        {
          "type": "Course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "Supports responsible automation of recurring analysis, reporting, and workflow tasks.",
          "eligibility": "Beginner course for business users; confirm current access and delivery options."
        }
      ]
    },
    {
      "id": "fpa_analyst",
      "title": "Financial Planning & Analysis Analyst",
      "summary": "Translate financial and operating performance into forecasts, scenarios, and decisions for business leaders.",
      "experienceTarget": 4.5,
      "skills": {
        "data_analytics": 0.9,
        "ai_automation": 0.7,
        "software_cloud": 0.52,
        "cybersecurity_risk": 0.4,
        "communication": 0.75,
        "project_change": 0.7,
        "creative_design": 0.25,
        "finance_commercial": 0.99,
        "people_coaching": 0.32,
        "operations_quality": 0.66,
        "customer_research": 0.42,
        "scientific_laboratory": 0.2
      },
      "coreSkills": [
        "finance_commercial",
        "data_analytics"
      ],
      "applicationContexts": [
        "Finance and insurance",
        "Retail and consumer goods",
        "Manufacturing",
        "Technology and telecommunications",
        "Professional services"
      ],
      "currentDemand": {
        "label": "Moderate",
        "score": 0.6,
        "basis": "Mixed evidence",
        "insight": "The Philippine services economy is large, but the PSA survey does not isolate FP&A. This signal is deliberately moderate because the reviewed national data is industry-level.",
        "sources": [
          {
            "name": "May 2026 Labor Force Survey",
            "owner": "Philippine Statistics Authority",
            "url": "https://psa.gov.ph/statistics/labor-force-survey?vcode=sl76S8",
            "published": "8 July 2026"
          }
        ]
      },
      "futureDemand": {
        "label": "Mixed",
        "score": 0.6,
        "basis": "Mixed evidence",
        "insight": "WEF expects accounting and audit roles to decline while analytical thinking remains a core skill. The opportunity is a move away from transaction processing toward scenarios, controls, and business partnering.",
        "sources": [
          {
            "name": "Future of Jobs 2025 — Jobs Outlook",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/2-jobs-outlook/",
            "published": "8 January 2025"
          },
          {
            "name": "Future of Jobs 2025 — Skills Outlook",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/3-skills-outlook/",
            "published": "8 January 2025"
          }
        ]
      },
      "aiOpportunity": "Use AI to draft scenarios, investigate variance, and summarize drivers while owning assumptions, controls, and advice to decision-makers.",
      "humanEdge": "Commercial judgment, governance, challenge of assumptions, and translating numbers into accountable choices.",
      "firstProof": "Create a driver-based forecast with three scenarios, sensitivity analysis, control checks, and a one-page management recommendation.",
      "certificationEvidence": {
        "name": "Certified Management Accountant (CMA)®",
        "issuer": "Institute of Management Accountants",
        "url": "https://www.imanet.org/ima-certifications/cma-certification",
        "eligibility": "Education, experience, membership, and two exam parts are required.",
        "why_it_fits": "CMA covers management accounting, planning, performance, analytics, controls, and strategic finance.",
        "practitioner": "Dylan Kady, CMA · Senior Financial Analyst",
        "practitioner_insight": "His issuer-published account reports applying pricing and forecasting learning and later receiving broader opportunities.",
        "source_type": "Issuer-published holder account",
        "practitioner_url": "https://www.imanet.org/en/Newsletters/Inside-IMA/2018/April/myCMA-Dylan-Kady",
        "caveat": "This is one issuer-published career story; education, experience, networking, and employer context also contributed."
      },
      "learningOptions": [
        {
          "type": "Certification",
          "name": "Certified Management Accountant (CMA)®",
          "provider": "Institute of Management Accountants",
          "url": "https://www.imanet.org/ima-certifications/cma-certification",
          "fit": "Covers planning, performance, analytics, controls, financial decision-making, and strategy.",
          "eligibility": "IMA requires membership, qualifying education, two years of relevant experience, and both exam parts."
        },
        {
          "type": "Certification",
          "name": "Certified Corporate FP&A Professional (FPAC)",
          "provider": "Association for Financial Professionals",
          "url": "https://fpacert.afponline.org/about-exam/eligibility",
          "fit": "Focuses on forecasting, modeling, planning, analytics, and business partnership.",
          "eligibility": "AFP applies education, relevant full-time experience, ethics, and two-part examination requirements."
        },
        {
          "type": "AI course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "A beginner, no-code course on applying generative AI to workflows, decisions, and business outcomes.",
          "eligibility": "Designed for business users across functions; confirm current product-access and delivery requirements."
        }
      ]
    },
    {
      "id": "laboratory_quality_specialist",
      "title": "Laboratory Quality & Scientific Operations Specialist",
      "summary": "Strengthen laboratory quality, method reliability, safety, compliance, and the operating systems that support defensible scientific results.",
      "experienceTarget": 3.5,
      "skills": {
        "data_analytics": 0.62,
        "ai_automation": 0.55,
        "software_cloud": 0.4,
        "cybersecurity_risk": 0.45,
        "communication": 0.68,
        "project_change": 0.58,
        "creative_design": 0.2,
        "finance_commercial": 0.18,
        "people_coaching": 0.3,
        "operations_quality": 0.85,
        "customer_research": 0.25,
        "scientific_laboratory": 0.99
      },
      "coreSkills": [
        "scientific_laboratory",
        "operations_quality"
      ],
      "applicationContexts": [
        "Pharmaceutical and healthcare laboratories",
        "Food and beverage testing",
        "Chemicals and manufacturing",
        "Environmental and government laboratories",
        "Research and testing services"
      ],
      "currentDemand": {
        "label": "Role-relevant",
        "score": 0.7,
        "basis": "Philippine regulatory evidence—not a vacancy count",
        "insight": "PRC states that only registered chemists can head a chemical analyses laboratory, supervise chemical work, and certify analyses in covered Philippine laboratories. This directly supports a regulated leadership path, but it does not quantify available jobs.",
        "sources": [
          {
            "name": "Chemistry Law requirements for chemical laboratories",
            "owner": "Philippine Professional Regulation Commission",
            "url": "https://prc.gov.ph/article/announcement-requirement-chemistry-law-certificate-authority-operate-chemical-laboratories",
            "published": "29 May 2017"
          }
        ]
      },
      "futureDemand": {
        "label": "Strong standards relevance",
        "score": 0.8,
        "basis": "Standards-based directional evidence—not an employment forecast",
        "insight": "ISO/IEC 17025 remains the international competence standard for testing and calibration laboratories. ISO notes its focus on competence, impartiality, consistent operation, information technology, and senior management responsibility. This supports continued quality-leadership relevance without proving future vacancy growth.",
        "sources": [
          {
            "name": "ISO/IEC 17025:2017 — Testing and calibration laboratories",
            "owner": "International Organization for Standardization",
            "url": "https://www.iso.org/standard/66912.html",
            "published": "Confirmed current in 2023; checked 28 July 2026"
          }
        ]
      },
      "aiOpportunity": "Use automation and AI to organize controlled documents, review structured quality data, detect trends, and draft investigation questions while keeping method validation, traceability, uncertainty, safety, and final scientific judgment under qualified human control.",
      "humanEdge": "Scientific accountability, laboratory safety, method suitability, traceability, ethical escalation, and responsibility for defensible results.",
      "firstProof": "Create a sanitized laboratory-improvement case: map one controlled process, identify a quality or safety risk, propose an evidence-based change, and define verification, documentation, and escalation controls.",
      "certificationEvidence": {
        "name": "Certified Quality Improvement Associate (CQIA)",
        "issuer": "American Society for Quality (ASQ)",
        "url": "https://www.asq.org/cert/quality-improvement-associate",
        "eligibility": "ASQ currently requires two years of full-time paid work experience, or an associate degree or two years of equivalent higher education. Confirm the current requirements and exam availability with ASQ.",
        "why_it_fits": "CQIA covers foundational quality tools, improvement methods, and teamwork. It can complement chemistry expertise when building evidence for laboratory-quality and process-improvement responsibilities.",
        "practitioner": "Krystel Sherman, ASQ CQIA · quality-control microbiologist and chemist",
        "practitioner_insight": "Her public professional profile lists CQIA alongside a chemistry and quality-control career. The available profile establishes credential use in a relevant profession but does not claim that CQIA caused a promotion or employment outcome.",
        "source_type": "Public credential-holder profile; no causal career outcome reported",
        "practitioner_url": "https://www.linkedin.com/in/krystel-sherman-asq-cqia-at-50235711",
        "caveat": "CQIA is a foundational quality credential, not a chemistry license, ISO/IEC 17025 laboratory accreditation, or proof of leadership. Philippine chemistry practice and laboratory-head responsibilities remain subject to PRC registration and applicable law."
      },
      "learningOptions": [
        {
          "type": "Certification",
          "name": "Certified Quality Improvement Associate (CQIA)",
          "provider": "American Society for Quality",
          "url": "https://www.asq.org/cert/quality-improvement-associate",
          "fit": "Builds foundational quality tools, team participation, and improvement-method knowledge.",
          "eligibility": "ASQ currently requires two years of full-time paid experience or qualifying higher education. Verify the current handbook."
        },
        {
          "type": "Course",
          "name": "Quality 101: CQIA Certification Preparation",
          "provider": "American Society for Quality",
          "url": "https://asq.org/training/quality-101--certified-quality-improvement-associate-certification-preparation-spcqia2020asq",
          "fit": "Official introductory quality course aligned with the CQIA body of knowledge.",
          "eligibility": "ASQ lists no prerequisite; course completion does not guarantee exam success."
        },
        {
          "type": "AI course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "A beginner, no-code course on applying generative AI to workflows, decisions, and business outcomes.",
          "eligibility": "Designed for business users across functions; confirm current product-access and delivery requirements."
        }
      ]
    },
    {
      "id": "customer_experience_manager",
      "title": "Customer Experience Manager",
      "summary": "Lead customer insight, journey improvement, service metrics, and cross-functional experience programs.",
      "experienceTarget": 5.5,
      "skills": {
        "data_analytics": 0.55,
        "ai_automation": 0.65,
        "software_cloud": 0.45,
        "cybersecurity_risk": 0.32,
        "communication": 0.97,
        "project_change": 0.8,
        "creative_design": 0.65,
        "finance_commercial": 0.35,
        "people_coaching": 0.85,
        "operations_quality": 0.72,
        "customer_research": 0.99,
        "scientific_laboratory": 0.15
      },
      "coreSkills": [
        "customer_research",
        "communication"
      ],
      "applicationContexts": [
        "Business-process and contact-center services",
        "Retail and e-commerce",
        "Banking and insurance",
        "Telecommunications",
        "Travel, hospitality, and healthcare"
      ],
      "currentDemand": {
        "label": "Strong",
        "score": 0.8,
        "basis": "Sector-to-role inference",
        "insight": "DOLE identifies IT–BPM/BPO and services as in-demand Philippine sectors. CX leadership is inferred from those sectors rather than counted as a standalone occupation.",
        "sources": [
          {
            "name": "Jobs and Labor Market Forecast",
            "owner": "DOLE Bureau of Local Employment",
            "url": "https://ble.dole.gov.ph/jobs-and-labor-market-forecast/",
            "published": "2023–2025 release"
          }
        ]
      },
      "futureDemand": {
        "label": "Mixed",
        "score": 0.6,
        "basis": "Mixed evidence",
        "insight": "GenAI can automate service tasks, but ILO's task-based evidence indicates transformation is more common than full job replacement. CX leadership becomes more valuable when it designs the human–AI handoff.",
        "sources": [
          {
            "name": "Generative AI and Jobs: A Refined Global Index",
            "owner": "International Labour Organization",
            "url": "https://www.ilo.org/publications/generative-ai-and-jobs-refined-global-index-occupational-exposure",
            "published": "20 May 2025"
          }
        ]
      },
      "aiOpportunity": "Use AI for conversation summaries, routing, and self-service while redesigning escalation, quality checks, accessibility, and recovery for high-stakes interactions.",
      "humanEdge": "Empathy, service recovery, organizational change, and accountability for customer trust.",
      "firstProof": "Map one service journey, identify safe automation points, and define handoff, quality, accessibility, and failure-recovery measures.",
      "certificationEvidence": {
        "name": "Certified Customer Experience Professional (CCXP)",
        "issuer": "Customer Experience Professionals Association",
        "url": "https://cxpaglobal.org/get-certified",
        "eligibility": "Education and multi-competency CX experience requirements apply.",
        "why_it_fits": "CCXP assesses strategy, insight, design, measurement, and culture across customer experience.",
        "practitioner": "Mariana De Marchi, CCXP · CX leader",
        "practitioner_insight": "She reports finding and securing a role through the CXPA community. Her account points to the network around the credential, not the exam alone.",
        "source_type": "Issuer-published holder account",
        "practitioner_url": "https://cxpaglobal.org/",
        "caveat": "The reported outcome is tied to association participation and community access; it should not be attributed solely to certification."
      },
      "learningOptions": [
        {
          "type": "Certification",
          "name": "Certified Customer Experience Professional (CCXP)",
          "provider": "Customer Experience Professionals Association",
          "url": "https://cxpaglobal.org/get-certified/get-started",
          "fit": "Validates experience across customer strategy, culture, insights, design, and measurement.",
          "eligibility": "CXPA lists education-and-experience routes; this is intended for experienced CX practitioners."
        },
        {
          "type": "Course directory",
          "name": "CXPA Recognized Training Providers",
          "provider": "Customer Experience Professionals Association",
          "url": "https://cxpaglobal.org/get-certified/recognized-training-providers",
          "fit": "Lists independently reviewed providers whose training aligns with the CXPA framework.",
          "eligibility": "Training is supplementary and does not replace CCXP experience or examination requirements."
        },
        {
          "type": "AI course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "A beginner, no-code course on applying generative AI to workflows, decisions, and business outcomes.",
          "eligibility": "Designed for business users across functions; confirm current product-access and delivery requirements."
        }
      ]
    },
    {
      "id": "ux_researcher",
      "title": "User Experience Researcher",
      "summary": "Study users, synthesize evidence, and help product teams design more useful and inclusive digital services.",
      "experienceTarget": 3.5,
      "skills": {
        "data_analytics": 0.65,
        "ai_automation": 0.72,
        "software_cloud": 0.58,
        "cybersecurity_risk": 0.3,
        "communication": 0.92,
        "project_change": 0.66,
        "creative_design": 0.98,
        "finance_commercial": 0.25,
        "people_coaching": 0.7,
        "operations_quality": 0.42,
        "customer_research": 0.98,
        "scientific_laboratory": 0.2
      },
      "coreSkills": [
        "customer_research",
        "creative_design",
        "communication"
      ],
      "applicationContexts": [
        "Software and digital products",
        "Finance and fintech",
        "Retail and e-commerce",
        "Healthcare technology",
        "Government and public digital services"
      ],
      "currentDemand": {
        "label": "Moderate",
        "score": 0.6,
        "basis": "Global directional evidence",
        "insight": "The reviewed Philippine sources do not provide a current UX researcher count. The signal relies on global evidence and is therefore kept moderate.",
        "sources": [
          {
            "name": "Future of Jobs 2025 — Industry Insights",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/5-region-economy-and-industry-insights/",
            "published": "8 January 2025"
          }
        ]
      },
      "futureDemand": {
        "label": "Mixed",
        "score": 0.6,
        "basis": "Mixed evidence",
        "insight": "WEF expects design and UX skills to rise overall, but some technology-services employers expect lower demand. Research rigor and AI-product evaluation are the more defensible specialization.",
        "sources": [
          {
            "name": "Future of Jobs 2025 — Skills Outlook",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/3-skills-outlook/",
            "published": "8 January 2025"
          },
          {
            "name": "Future of Jobs 2025 — Industry Insights",
            "owner": "World Economic Forum",
            "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/5-region-economy-and-industry-insights/",
            "published": "8 January 2025"
          }
        ]
      },
      "aiOpportunity": "Use AI to accelerate recruiting materials, synthesis, and prototypes while preserving research validity, consent, accessibility, and direct contact with users.",
      "humanEdge": "Choosing the right question, noticing weak evidence, facilitating difficult conversations, and representing affected users.",
      "firstProof": "Run a five-user study of an AI feature, document consent and limitations, and show which product decision changed because of evidence.",
      "certificationEvidence": {
        "name": "NN/g UX Certification",
        "issuer": "Nielsen Norman Group",
        "url": "https://www.nngroup.com/ux-certification/",
        "eligibility": "Five eligible live courses and corresponding exams are required.",
        "why_it_fits": "The program can combine research, design, and AI-related courses and publishes its requirements and cost.",
        "practitioner": "Corey Nunez · UX-certified practitioner",
        "practitioner_insight": "His issuer-published testimonial says the credential added credibility to his decisions in industry.",
        "source_type": "Issuer-published holder account",
        "practitioner_url": "https://www.nngroup.com/ux-certification/",
        "caveat": "NN/g states that the program is not accredited. Holder comments are selected testimonials and the listed investment is substantial."
      },
      "learningOptions": [
        {
          "type": "Professional certificate",
          "name": "Google UX Design Certificate",
          "provider": "Google",
          "url": "https://grow.google/certificates/ux-design/",
          "fit": "Covers user research, accessibility, wireframes, prototypes, usability testing, and portfolio work.",
          "eligibility": "Google describes it as foundational and requiring no previous experience."
        },
        {
          "type": "Certification",
          "name": "UX Certification",
          "provider": "Nielsen Norman Group",
          "url": "https://www.nngroup.com/ux-certification/",
          "fit": "Course- and exam-based professional development across UX research and design topics.",
          "eligibility": "Review the current course-count, examination, pricing, and delivery requirements directly with NN/g."
        },
        {
          "type": "AI course",
          "name": "Transform business workflows with generative AI",
          "provider": "Microsoft Learn",
          "url": "https://learn.microsoft.com/en-us/training/courses/ab-730t00",
          "fit": "A beginner, no-code course on applying generative AI to workflows, decisions, and business outcomes.",
          "eligibility": "Designed for business users across functions; confirm current product-access and delivery requirements."
        }
      ]
    }
  ],
  "sourceLinks": [
    {
      "name": "May 2026 Labor Force Survey",
      "owner": "Philippine Statistics Authority",
      "url": "https://psa.gov.ph/statistics/labor-force-survey?vcode=sl76S8",
      "published": "8 July 2026"
    },
    {
      "name": "Jobs and Labor Market Forecast",
      "owner": "DOLE Bureau of Local Employment",
      "url": "https://ble.dole.gov.ph/jobs-and-labor-market-forecast/",
      "published": "2023–2025 release"
    },
    {
      "name": "TVET Skills Insights: 5th Industrial Revolution",
      "owner": "TESDA",
      "url": "https://www.tesda.gov.ph/Uploads/File/SkillInsights/2025/TVET%20Skills%20Insights%20Report%20_%205th%20Industrial%20Revolution.pdf",
      "published": "2025"
    },
    {
      "name": "Future of Jobs 2025 — Jobs Outlook",
      "owner": "World Economic Forum",
      "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/2-jobs-outlook/",
      "published": "8 January 2025"
    },
    {
      "name": "Future of Jobs 2025 — Skills Outlook",
      "owner": "World Economic Forum",
      "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/3-skills-outlook/",
      "published": "8 January 2025"
    },
    {
      "name": "Future of Jobs 2025 — Industry Insights",
      "owner": "World Economic Forum",
      "url": "https://www.weforum.org/publications/the-future-of-jobs-report-2025/in-full/5-region-economy-and-industry-insights/",
      "published": "8 January 2025"
    },
    {
      "name": "Generative AI and Jobs: A Refined Global Index",
      "owner": "International Labour Organization",
      "url": "https://www.ilo.org/publications/generative-ai-and-jobs-refined-global-index-occupational-exposure",
      "published": "20 May 2025"
    },
    {
      "name": "PMP salary and talent survey",
      "owner": "Project Management Institute",
      "url": "https://www.pmi.org/about/press-media/2025/pmp-certification-holders-build-career-momentum-and-experience-earning-advantage-pmi-survey-finds",
      "published": "13 November 2025"
    },
    {
      "name": "Who earns the ISC2 CC?",
      "owner": "ISC2",
      "url": "https://www.isc2.org/insights/2025/11/who-earns-the-isc2-certified-in-cybersecurity-certification",
      "published": "3 November 2025"
    },
    {
      "name": "People Analytics Specialty Credential",
      "owner": "Society for Human Resource Management",
      "url": "https://www.shrm.org/credentials/specialty-credentials/people-analytics-credential",
      "published": "Checked 28 July 2026"
    },
    {
      "name": "Certified in Planning and Inventory Management",
      "owner": "Association for Supply Chain Management",
      "url": "https://www.ascm.org/learning-development/certifications-credentials/cpim/",
      "published": "Checked 28 July 2026"
    },
    {
      "name": "Chemistry Law requirements for chemical laboratories",
      "owner": "Philippine Professional Regulation Commission",
      "url": "https://prc.gov.ph/article/announcement-requirement-chemistry-law-certificate-authority-operate-chemical-laboratories",
      "published": "29 May 2017"
    },
    {
      "name": "ISO/IEC 17025:2017 — Testing and calibration laboratories",
      "owner": "International Organization for Standardization",
      "url": "https://www.iso.org/standard/66912.html",
      "published": "Confirmed current in 2023; checked 28 July 2026"
    }
  ]
}

In [ ]:
SKILLS = [item["key"] for item in CATALOG["skills"]]
SKILL_LABELS = {
    item["key"]: item["label"] for item in CATALOG["skills"]
}
CAREERS = {item["id"]: item for item in CATALOG["careers"]}
FEATURES = ["years_experience", *SKILLS]

print("Features:", FEATURES)
print("Excluded:", CATALOG["excludedFeatures"])
print("Job families:", len(CAREERS))

## Generate 2,200 reproducible synthetic profiles

There are 200 profiles per job family. Some profiles blend with an
adjacent prototype so classes overlap rather than being unrealistically
perfect. A skill value of **0 means no experience**.

In [ ]:
def clip_rating(value):
    return int(np.clip(np.rint(value), 1, 5))


def generate_synthetic_profiles(per_career=200):
    rows = []
    career_ids = list(CAREERS)
    profile_number = 1
    for career_index, career_id in enumerate(career_ids):
        prototype = CAREERS[career_id]
        adjacent = CAREERS[
            career_ids[(career_index + 1) % len(career_ids)]
        ]
        target = [prototype["skills"][skill] for skill in SKILLS]
        adjacent_target = [
            adjacent["skills"][skill] for skill in SKILLS
        ]

        for _ in range(per_career):
            blend = (
                float(RNG.uniform(0.08, 0.30))
                if RNG.random() < 0.32
                else 0.0
            )
            experience_center = (
                prototype["experienceTarget"] * (1 - blend)
                + adjacent["experienceTarget"] * blend
            )
            row = {
                "profile_id": f"SKILL-{profile_number:04d}",
                "years_experience": round(
                    float(
                        np.clip(
                            RNG.normal(experience_center, 2.4),
                            0,
                            30,
                        )
                    ),
                    1,
                ),
            }
            for index, skill in enumerate(SKILLS):
                center = (
                    target[index] * (1 - blend)
                    + adjacent_target[index] * blend
                )
                zero_chance = 0.02 + 0.18 * (1 - center) ** 2
                row[skill] = (
                    0
                    if RNG.random() < zero_chance
                    else clip_rating(
                        1 + 4 * center + RNG.normal(0, 0.58)
                    )
                )
            row["recommended_career"] = career_id
            rows.append(row)
            profile_number += 1

    frame = pd.DataFrame(rows)
    assert len(frame) == 2200
    assert frame["recommended_career"].value_counts().eq(200).all()
    return frame


synthetic_profiles = generate_synthetic_profiles()
display(synthetic_profiles.head())
display(
    synthetic_profiles["recommended_career"]
    .value_counts()
    .rename("profiles")
    .to_frame()
)

## Compare candidate classifiers

Extra Trees, Random Forest, multinomial logistic regression, and
distance-weighted 7-nearest-neighbors are compared with stratified
five-fold accuracy and macro-F1. The winner is chosen by macro-F1.

In [ ]:
X = synthetic_profiles[FEATURES]
y = synthetic_profiles["recommended_career"]
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

candidates = {
    "Extra Trees": Pipeline([
        ("scale", StandardScaler()),
        ("model", ExtraTreesClassifier(
            n_estimators=500,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=SEED,
            n_jobs=-1,
        )),
    ]),
    "Random Forest": Pipeline([
        ("scale", StandardScaler()),
        ("model", RandomForestClassifier(
            n_estimators=500,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=SEED,
            n_jobs=-1,
        )),
    ]),
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            random_state=SEED,
        )),
    ]),
    "Distance-weighted 7-NN": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=7,
            weights="distance",
        )),
    ]),
}

benchmark_rows = []
for name, candidate in candidates.items():
    accuracy_scores, macro_f1_scores = [], []
    for train_index, test_index in folds.split(X, y):
        candidate.fit(X.iloc[train_index], y.iloc[train_index])
        prediction = candidate.predict(X.iloc[test_index])
        accuracy_scores.append(
            accuracy_score(y.iloc[test_index], prediction)
        )
        macro_f1_scores.append(
            f1_score(
                y.iloc[test_index],
                prediction,
                average="macro",
            )
        )
    benchmark_rows.append({
        "model": name,
        "mean_accuracy": np.mean(accuracy_scores),
        "mean_macro_f1": np.mean(macro_f1_scores),
    })

benchmark = (
    pd.DataFrame(benchmark_rows)
    .sort_values(
        ["mean_macro_f1", "mean_accuracy"],
        ascending=False,
    )
    .reset_index(drop=True)
)
display(benchmark.style.format({
    "mean_accuracy": "{:.3f}",
    "mean_macro_f1": "{:.3f}",
}))
print(
    "Reminder: these scores describe synthetic-label recovery only."
)

In [ ]:
model = candidates[benchmark.loc[0, "model"]]
model.fit(X, y)
print("Selected model:", benchmark.loc[0, "model"])

## Evidence-aware recommendation function

The final comparative score uses:

- skill alignment: 50%;
- core-skill coverage: 15%;
- synthetic model fit: 15%;
- experience proximity: 8%;
- research-graded current demand: 5%;
- research-graded future demand: 7%.

A job is withheld unless it clears all support thresholds. The function
returns at most three jobs and may return fewer.

In [ ]:
def cosine(left, right):
    denominator = np.linalg.norm(left) * np.linalg.norm(right)
    return float(np.dot(left, right) / denominator) if denominator else 0


def recommend_jobs(years_experience, skill_ratings, top_k=3):
    if not 0 <= years_experience <= 50:
        raise ValueError("Years of experience must be between 0 and 50.")
    missing = [skill for skill in SKILLS if skill not in skill_ratings]
    if missing:
        raise ValueError(f"Missing skill ratings: {', '.join(missing)}")
    if any(not 0 <= skill_ratings[skill] <= 5 for skill in SKILLS):
        raise ValueError("Every skill rating must be between 0 and 5.")
    if not any(skill_ratings[skill] > 0 for skill in SKILLS):
        raise ValueError(
            "Rate at least one skill above 0 so the model has evidence."
        )

    row = {"years_experience": years_experience, **skill_ratings}
    probabilities = dict(
        zip(model.classes_, model.predict_proba(pd.DataFrame([row]))[0])
    )
    user = np.array([skill_ratings[skill] / 5 for skill in SKILLS])
    results = []

    for career_id, career in CAREERS.items():
        target = np.array(
            [career["skills"][skill] for skill in SKILLS]
        )
        alignment = (
            0.65 * cosine(user, target)
            + 0.35 * np.minimum(user, target).sum() / target.sum()
        )
        core_coverage = np.mean([
            min(
                1,
                (skill_ratings[skill] / 5)
                / max(0.20, career["skills"][skill]),
            )
            for skill in career["coreSkills"]
        ])
        experience_fit = np.exp(
            -abs(
                years_experience - career["experienceTarget"]
            )
            / max(3, career["experienceTarget"] * 0.8)
        )
        current = career["currentDemand"]["score"]
        future = career["futureDemand"]["score"]
        score = 100 * (
            0.50 * alignment
            + 0.15 * core_coverage
            + 0.15 * probabilities.get(career_id, 0)
            + 0.08 * experience_fit
            + 0.05 * current
            + 0.07 * future
        )
        gaps = sorted(
            [
                {
                    "skill": SKILL_LABELS[skill],
                    "current": skill_ratings[skill],
                    "target": int(
                        np.clip(
                            np.rint(5 * career["skills"][skill]),
                            1,
                            5,
                        )
                    ),
                }
                for skill in SKILLS
                if skill_ratings[skill]
                < np.rint(5 * career["skills"][skill])
            ],
            key=lambda item: item["target"] - item["current"],
            reverse=True,
        )[:4]
        results.append({
            "career_id": career_id,
            "job": career["title"],
            "comparative_score": round(score, 1),
            "skill_alignment": round(100 * alignment, 1),
            "core_skill_coverage": round(100 * core_coverage, 1),
            "summary": career["summary"],
            "skill_gaps": gaps,
            "ai_opportunity": career["aiOpportunity"],
            "human_edge": career["humanEdge"],
            "portfolio_proof": career["firstProof"],
            "current_demand": career["currentDemand"],
            "future_demand": career["futureDemand"],
            "learning_options": career["learningOptions"],
            "application_contexts": career["applicationContexts"],
            "credential_holder_account": (
                career["certificationEvidence"]
            ),
        })

    supported = [
        item for item in results
        if item["skill_alignment"] >= 40
        and item["core_skill_coverage"] >= 25
        and item["comparative_score"] >= 55
    ]
    return sorted(
        supported,
        key=lambda item: item["comparative_score"],
        reverse=True,
    )[:top_k]

## Example: a chemist/scientific profile

The example deliberately does **not** pass “chemist,” “healthcare,” or
“move toward leadership.” Its scientific and quality skills must carry
the recommendation.

In [ ]:
example = dict.fromkeys(SKILLS, 0)
example.update({
    "communication": 3,
    "finance_commercial": 1,
    "operations_quality": 1,
    "customer_research": 3,
    "scientific_laboratory": 5,
})

recommendations = recommend_jobs(
    years_experience=3,
    skill_ratings=example,
)
print([item["job"] for item in recommendations])
display(pd.json_normalize(recommendations))

## How to interpret and improve the model

- Treat each result as a hypothesis to investigate, not an instruction.
- Read the linked evidence and confirm current eligibility before paying
  for any course or exam.
- A practitioner account is not causal proof that a credential creates a
  job outcome.
- Replace synthetic profiles with consented, de-identified, outcome-linked
  records before making production performance claims.
- Add occupation taxonomy, current job-posting validation, geographic
  coverage, calibration, subgroup fairness tests, and human career-adviser
  review before production use.